<a href="https://colab.research.google.com/github/abdulqadoos861/GetData_practice_ML/blob/main/pipelines.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

importing All packages used during this project.


In [118]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import OrdinalEncoder
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import accuracy_score
from sklearn.tree import DecisionTreeClassifier
from sklearn.pipeline import Pipeline , make_pipeline
from sklearn.feature_selection import SelectKBest , chi2

print("All Libraries are imported successfully...")



All Libraries are imported successfully...


In [119]:
df = sns.load_dataset("titanic")
df

,survived,pclass,sex,age,sibsp,parch,fare,embarked,class,who,adult_male,deck,embark_town,alive,alone
0,0,3,male,22.0,1,0,7.2500,S,Third,man,True,NaN,Southampton,no,False
1,1,1,female,38.0,1,0,71.2833,C,First,woman,False,C,Cherbourg,yes,False
2,1,3,female,26.0,0,0,7.9250,S,Third,woman,False,NaN,Southampton,yes,True
3,1,1,female,35.0,1,0,53.1000,S,First,woman,False,C,Southampton,yes,False
4,0,3,male,35.0,0,0,8.0500,S,Third,man,True,NaN,Southampton,no,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
886,0,2,male,27.0,0,0,13.0000,S,Second,man,True,NaN,Southampton,no,True
887,1,1,female,19.0,0,0,30.0000,S,First,woman,False,B,Southampton,yes,True
888,0,3,female,NaN,1,2,23.4500,S,Third,woman,False,NaN,Southampton,no,False
889,1,1,male,26.0,0,0,30.0000,C,First,man,True,C,Cherbourg,yes,True


# Lets Plan...


In [120]:
# Drop unnecessary Columns...
df.drop(columns=["class","who","adult_male","deck","embark_town"	,"alive","alone"],inplace=True)

In [121]:
# split Train and test data ...
X_train , X_test , Y_train , Y_test = train_test_split(df.drop(columns=['survived']) , df["survived"] , test_size=0.2 , random_state=1)

In [122]:
# Imputation Transofmer
trf1 = ColumnTransformer([
    ("impute_age" , SimpleImputer() , [2]),
    ("impute_embarked" , SimpleImputer(strategy='most_frequent') , [6])
] , remainder="passthrough")

In [123]:
# One hot encoding because here targeted columns are minimal . if is there any column are ordinal than i should use ordinalEnocder for that
trf2 = ColumnTransformer([
    ("one_hot_sex_encoding" , OneHotEncoder(sparse_output=False , handle_unknown="ignore"),[1,6] )
] , remainder="passthrough")

In [124]:
# Scalling
trf3 = ColumnTransformer([
    ('scaler' , MinMaxScaler() , slice(0,10))
])

In [125]:
# Feature Selection
trf4 = SelectKBest(score_func=chi2 , k=5)

In [126]:
# Train_model
trf5= DecisionTreeClassifier()

# Creating Pipline

In [127]:
pipe = Pipeline([
    ("trf1" , trf1),
    ("trf2" , trf2),
    ("trf3" , trf3),
    # ("trf4" , trf4),
    ("trf5" , trf5)
])

# **Pipeline** Vs **Make_pipline**

Both are used to create pipline but Pipline() class requires object and its name , while make_pipline method only requires only object names .

In [128]:
# pipe alternate syntax
pipe2 = make_pipeline(trf1 , trf2 , trf3 , trf4 , trf5)

In [129]:
# train
pipe.fit(X_train , Y_train)

Pipeline(steps=[('trf1',
                 ColumnTransformer(remainder='passthrough',
                                   transformers=[('impute_age', SimpleImputer(),
                                                  [2]),
                                                 ('impute_embarked',
                                                  SimpleImputer(strategy='most_frequent'),
                                                  [6])])),
                ('trf2',
                 ColumnTransformer(remainder='passthrough',
                                   transformers=[('one_hot_sex_encoding',
                                                  OneHotEncoder(handle_unknown='ignore',
                                                                sparse_output=False),
                                                  [1, 6])])),
                ('trf3',
                 ColumnTransformer(transformers=[('scaler', MinMaxScaler(),
                                                  slice(0, 10, None))])),
                ('trf5', DecisionTreeClassifier())])

Now our pipe is trained and now we have to predict using pipe.

In [130]:
y_pred = pipe.predict(X_test)

In [131]:
accuracy_score(Y_test , y_pred)

0.6201117318435754

# Cross Validation

In [132]:
# Cross Validation Using Cross_Val_Score
from sklearn.model_selection import cross_val_score
cross_val_score(pipe , X_train ,Y_train , cv = 5 , scoring="accuracy").mean()

np.float64(0.6404510981975771)

# Grid Search

In [135]:
# Grid Search cv
params = {
    'trf5__max_depth' : [1,2,3,4,5,None]
}

In [136]:
from sklearn.model_selection import GridSearchCV
grid = GridSearchCV(pipe , params , cv = 5 , scoring="accuracy")
grid.fit(X_train , Y_train)

GridSearchCV(cv=5,
             estimator=Pipeline(steps=[('trf1',
                                        ColumnTransformer(remainder='passthrough',
                                                          transformers=[('impute_age',
                                                                         SimpleImputer(),
                                                                         [2]),
                                                                        ('impute_embarked',
                                                                         SimpleImputer(strategy='most_frequent'),
                                                                         [6])])),
                                       ('trf2',
                                        ColumnTransformer(remainder='passthrough',
                                                          transformers=[('one_hot_sex_encoding',
                                                                         OneHotEncoder(handle_unknown='ignore',
                                                                                       sparse_output=False),
                                                                         [1,
                                                                          6])])),
                                       ('trf3',
                                        ColumnTransformer(transformers=[('scaler',
                                                                         MinMaxScaler(),
                                                                         slice(0, 10, None))])),
                                       ('trf5', DecisionTreeClassifier())]),
             param_grid={'trf5__max_depth': [1, 2, 3, 4, 5, None]},
             scoring='accuracy')

Exporting File

In [137]:
import pickle
pickle.dump(pipe , open("pipe.pkl" , "wb"))
#

Loading pickle

In [138]:
pipe_pro = pickle.load(open("pipe.pkl" , "rb"))

In [141]:
#  Create some sample test data
# Ensure the columns match the input features used for training (X_train, X_test)
sample_data = {
    'pclass': [1, 3, 2, 3],
    'sex': ['male', 'female', 'male', 'female'],
    'age': [30.0, 20.0, 45.0, 5.0],
    'sibsp': [1, 0, 0, 2],
    'parch': [0, 1, 0, 1],
    'fare': [50.0, 15.0, 10.0, 25.0],
    'embarked': ['S', 'C', 'S', 'Q']
}

new_X_test = pd.DataFrame(sample_data)

In [140]:
# Make predictions on the new sample data using the loaded pipeline
new_predictions = pipe_pro.predict(new_X_test)

print("Predictions for new test data:")
print(new_predictions)

Predictions for new test data:
[0 1 0 0]
